# 🏗️ Notebook 1: Flash Sale — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/flash-sale
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window:
`Cmd+Shift+P` → **Reload Window**.

This lab uses **only pure Python** (plus `pydantic`) — no Docker, no external services.
We simulate Redis, queues, and threads in-process so every cell runs anywhere.


## 🎯 What is a flash sale?

> **Flash sale:** a tiny amount of stock goes on sale at a fixed moment, and a huge crowd
> tries to buy it all in the first few seconds.

Concrete examples from the real world:

| Event | Scale |
|---|---|
| **Alibaba Singles' Day (11/11)** | ~583,000 orders / second at peak (2020) |
| **Xiaomi phone drops** | Millions of users, thousands of phones, sells out in seconds |
| **Supreme Thursday drops** | Tiny stock, fans refreshing for minutes before 11:00 AM ET |
| **PS5 / GPU launches** | Worldwide scalpers + bots, site crashes common |
| **Taylor Swift Eras tour (Ticketmaster 2022)** | ~14M users for 2M tickets — site melted, stayed down for hours |

The engineering challenge is identical in each case:

1. A wave of traffic orders of magnitude bigger than normal.
2. A finite, **small** amount of inventory.
3. Strong correctness rule: **never sell more than you have**.
4. Mostly-losers: 99%+ of users will *not* get anything — but they must get a fast answer.


## 📋 Requirements

### Functional
- Sale starts at a fixed `start_at` timestamp. Before that: page is unavailable.
- Hard cap on stock per SKU (e.g., 1,000 units of `item-42`).
- Per-user cap (e.g., 1 or 2 items per user_id / per payment method / per shipping address).
- Reserve stock first, **charge later** (payment is slow, stock is scarce).
- Reservation expires (TTL) if not paid → stock returns to pool.

### Non-functional
- **Spike traffic**: 100k–1M RPS for a few seconds, then back to normal.
- **No overselling** under any concurrency.
- **Fast "no"**: losers must get their 409/sold-out page within a second.
- **Fairness**: roughly "who arrived first gets a chance" (not literal global FIFO).
- **Bot resistance**: bots will try to win everything; raise their cost.


## 🧮 Back-of-the-envelope math

Let's turn "a huge crowd" into numbers you can design against.
Running this cell makes the constraints concrete.


In [1]:
def capacity_plan(users: int, stock: int, sale_window_s: int = 5,
                  req_per_user: int = 3, rps_per_redis: int = 100_000):
    """Rough sizing for a flash sale.

    Parameters
    ----------
    users : total concurrent users hitting refresh at t=0
    stock : total units for sale
    sale_window_s : how long the sale realistically lasts (usually 1-10s)
    req_per_user : retries + page loads per user
    rps_per_redis : ballpark ops/sec a single Redis primary can sustain
    """
    total_reqs = users * req_per_user
    peak_rps = total_reqs / sale_window_s
    win_rate = stock / users
    redis_shards = max(1, int(peak_rps / rps_per_redis) + 1)

    print(f"Users          : {users:>12,}")
    print(f"Stock          : {stock:>12,}")
    print(f"Total requests : {total_reqs:>12,}  ({req_per_user} per user)")
    print(f"Peak RPS       : {peak_rps:>12,.0f}  (over {sale_window_s}s)")
    print(f"Win rate       : {win_rate:>12.4%}  (99%+ of users get nothing)")
    print(f"Redis shards   : {redis_shards:>12}  (at ~{rps_per_redis:,} ops/s each)")
    return peak_rps

# Example: 1M users fighting for 1,000 items
capacity_plan(users=1_000_000, stock=1_000)


Users          :    1,000,000
Stock          :        1,000
Total requests :    3,000,000  (3 per user)
Peak RPS       :      600,000  (over 5s)
Win rate       :      0.1000%  (99%+ of users get nothing)
Redis shards   :            7  (at ~100,000 ops/s each)


600000.0

### What the numbers tell us

- **Peak RPS dwarfs steady-state.** You can't provision for this 24/7 — you need elastic scaling
  and a way to *shed* load quickly (waiting room, rate limits, bounded queue).
- **99.9% of users will lose.** Optimize the "you missed out" path: it has to be **static**,
  cheap, and served from a CDN. You do not want a million DB queries to tell people "sorry."
- **Single Redis won't cut it** at 1M RPS. You'll need sharding or multi-primary — but at small
  scales (<100k RPS) a single Redis + Lua is plenty.


## 🏛️ Architecture (top-down)

```
   [ 1,000,000 clients ]
           │
           ▼   static page until start_at
   ┌───────────────────┐
   │  CDN / Edge       │  ← "sale not open yet" HTML, cached, no origin hit
   └─────────┬─────────┘
             │  at start_at, CDN starts serving the "click to enter" page
             ▼
   ┌───────────────────┐
   │  Waiting Room     │  ← cryptographic token says "you may proceed at t=X"
   └─────────┬─────────┘
             │
             ▼
   ┌───────────────────┐
   │  Rate Limiter     │  ← token bucket per user_id / IP
   │  (edge, cheap)    │
   └─────────┬─────────┘
             │
             ▼
   ┌───────────────────┐
   │  Admission Queue  │  ← bounded. If full → 503 fast. Drains at a fixed rate.
   │  (Kafka / Redis)  │
   └─────────┬─────────┘
             │ consumers pull at safe rate
             ▼
   ┌────────────────────────┐
   │  Stock Service         │  atomic reserve (Redis Lua / DECR)
   │  Redis: stock:item-42  │
   └─────────┬──────────────┘
             │ reserved  →  reservation:{id} with TTL (e.g. 10 min)
             ▼
   ┌───────────────────┐     ┌───────────────────┐
   │  Payment Service  │────▶│  Orders DB        │
   │  (slow, external) │     │  (PostgreSQL)     │
   └───────────────────┘     └───────────────────┘
```

### Why each layer exists

| Layer | Problem it solves |
|---|---|
| **CDN / static page** | Stops 1M users from hitting origin *before* the sale opens. |
| **Waiting room** | Smooths the instantaneous t=0 spike into a controlled stream. |
| **Rate limiter** | One user ≠ one request; punishes bots hammering the endpoint. |
| **Bounded queue** | Backpressure. Once full, we know the sale is effectively over → fast "no." |
| **Atomic stock (Redis)** | The only place that knows "how many left." Must be atomic. |
| **Async payment** | Payments can take seconds. Don't hold stock in flight — reserve then charge. |
| **Orders DB** | Durable record. Populated *after* the chaos, from reservations. |


## 🧠 The 5 core ideas (remember these)

1. **Push the crowd far from the origin.** CDN and waiting room absorb millions of clients
   with static bytes. The API never sees most of them.
2. **Fail fast.** Losers deserve a 1-second "sold out" page, not a 30-second timeout.
3. **Make stock atomic in one place.** Redis key + Lua. Not MySQL, not microservices plural.
4. **Separate reservation from payment.** Reserve is cheap and synchronous; paying is slow
   and async. If payment fails or times out, reservation TTL returns the stock.
5. **Be honest about fairness.** Global FIFO at planetary scale is a lie. "Best-effort FIFO
   by queue arrival" is the realistic goal.

In the next notebook we'll nail down the **data model** and the **APIs**. In notebook 3 we'll
actually implement the hot path and show a **bad → best** progression.
